# Nettoyage et dédoublonnage de l'agrégat SFT

Objectif : exécuter et documenter le nettoyage appliqué par `clean_sft_dataset` (dans `scripts/extraction.py`) sur l'agrégat SFT (MediQAl + FrenchMedMCQA + MedQuAD). L'exploration qui a mené à cette logique est dans `notebooks/01_exploration_sources.ipynb` : 48 doublons exacts repérés dans MedQuAD, 1 doublon exact dans FrenchMedMCQA, aucun dans MediQAl.

Ce notebook ne redéfinit pas la logique de nettoyage, il l'exécute et montre le résultat, pour garder une trace des transformations appliquées (auditabilité RGPD, voir `docs/etape1.md`).

In [1]:
import sys
sys.path.append("..")

from collections import Counter
from dotenv import load_dotenv
from scripts.extraction import (
    load_mediqa,
    load_frenchmedmcqa,
    load_medquad,
    clean_sft_dataset,
    build_sft_dataset,
)
load_dotenv()

/home/rapha/ia-engineer/llm-finetuning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

## MedQuAD : 48 doublons exacts

48 lignes partagent exactement le même couple (question, réponse) que la ligne `instruction`/`reponse` construite dans `load_medquad`. On les retrouve ici avec la même méthode que dans le notebook d'exploration.

In [2]:
medquad = load_medquad()
print("medquad brut :", len(medquad))

cles = [(r["instruction"], r["reponse"]) for r in medquad]
compteur = Counter(cles)
doublons = {cle: n for cle, n in compteur.items() if n > 1}
print("couples (instruction, reponse) en double :", len(doublons))
print("lignes en trop à retirer :", sum(n - 1 for n in doublons.values()))

medquad brut : 16407
couples (instruction, reponse) en double : 32
lignes en trop à retirer : 48


Trois exemples de doublons, pour voir à quoi ils ressemblent :

In [3]:
for instruction, reponse in list(doublons.keys())[:3]:
    print("instruction :", instruction)
    print("reponse     :", reponse[:150].replace("\n", " "), "...")
    print()

instruction : What is (are) Hypoglycemia ?
reponse     : Hypoglycemia, also called low blood glucose or low blood sugar, occurs when blood glucose drops below normal levels. Glucose, an important source of e ...

instruction : What are the symptoms of Hypoglycemia ?
reponse     : Hypoglycemia causes symptoms such as                  - hunger  - shakiness  - nervousness  - sweating  - dizziness or light-headedness  - sleepiness  ...

instruction : What causes Hypoglycemia ?
reponse     : Diabetes Medications                  Hypoglycemia can occur as a side effect of some diabetes medications, including insulin and oral diabetes medica ...



Comme repéré dans le notebook 01, ce sont des questions génériques qui reviennent (« What is (are) Hypoglycemia ? » posée deux fois avec la même réponse), pas des variantes à garder : ce sont bien des doublons exacts à retirer.

## FrenchMedMCQA : 1 doublon exact

Un seul doublon, déjà repéré dans le notebook d'exploration : une question sur Streptococcus pneumoniae apparaît deux fois dans le split train, avec le même id.

In [4]:
frenchmedmcqa = load_frenchmedmcqa()
print("frenchmedmcqa brut :", len(frenchmedmcqa))

cles = [(r["instruction"], r["reponse"]) for r in frenchmedmcqa]
compteur = Counter(cles)
doublons = {cle: n for cle, n in compteur.items() if n > 1}
print("couples en double :", len(doublons))

for instruction, reponse in doublons.keys():
    print("instruction :", instruction)
    print("reponse     :", reponse)

frenchmedmcqa brut : 1080
couples en double : 1
instruction : Parmi les caractères suivants, lequel n'est pas retrouvé chez Streptococcus pneumoniae ?
a) Présence d'une capsule
b) Catalase négative
c) Survie en aérobiose et anaérobiose
d) Production d'une toxine de Panton-Valentine
e) Hémolytique sur gélose au sang
reponse     : d) Production d'une toxine de Panton-Valentine


## MediQAl : aucun doublon

Confirmation sur l'agrégat construit par `load_mediqa`, cohérente avec le notebook d'exploration (qui incluait déjà le cas clinique dans la comparaison).

In [5]:
mediqa = load_mediqa()
print("mediqa brut :", len(mediqa))

cles = [(r["instruction"], r["reponse"]) for r in mediqa]
print("couples uniques :", len(set(cles)))

mediqa brut : 4969
couples uniques : 4969


## Agrégat final

`build_sft_dataset` charge les trois sources puis applique `clean_sft_dataset`, qui retire les doublons exacts (instruction, reponse) en gardant la première occurrence rencontrée.

In [6]:
agregat_brut = mediqa + frenchmedmcqa + medquad
agregat_nettoye = build_sft_dataset()

print("agrégat brut    :", len(agregat_brut))
print("agrégat nettoyé :", len(agregat_nettoye))
print("doublons retirés :", len(agregat_brut) - len(agregat_nettoye))

retires_par_source = Counter()
vus = set()
for r in agregat_brut:
    cle = (r["instruction"], r["reponse"])
    if cle in vus:
        retires_par_source[r["source"]] += 1
    else:
        vus.add(cle)
print("répartition des doublons retirés :", dict(retires_par_source))

agrégat brut    : 22456
agrégat nettoyé : 22407
doublons retirés : 49
répartition des doublons retirés : {'frenchmedmcqa': 1, 'medquad': 48}


## Synthèse

22456 exemples SFT bruts, 49 doublons exacts retirés (48 dans MedQuAD, 1 dans FrenchMedMCQA), 22407 exemples dans l'agrégat nettoyé. Aucun doublon cross-source détecté, ce qui est attendu vu que les langues et les sujets diffèrent d'une source à l'autre.

Prochaines étapes sur cet agrégat : définir les splits train / validation / test / eval_clinique (MediQAl et MedQuAD n'ont pas de split hérité de la source), puis sous-échantillonner à environ 5000 paires, puis anonymiser avec Presidio.